In [2]:
# Install dependencies ──
!pip install scikit-learn pandas matplotlib seaborn

# **TF-IDF + SVM**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import requests

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")


# import data
df = pd.read_csv('patient queries.csv')

In [4]:
QUESTION_COL = "Question"
INTENT_COL   = "Manual Validation"

df = df[[QUESTION_COL, INTENT_COL]].dropna()
df.columns = ["text", "label"]

# Basic text cleaning
df["text"] = (
    df["text"]
    .str.lower()
    .str.strip()
    .str.replace(r'["\']', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

print("\nClass distribution:")
print(df["label"].value_counts())
print(f"\nTotal samples : {len(df)}")
print(f"Unique intents: {df['label'].nunique()}")


Class distribution:
label
GET_WARNING_PRECAUTIONS            202
GET_ADMINISTRATION_INSTRUCTIONS    200
GET_SIDE_EFFECTS                   200
GET_STORAGE_INSTRUCTIONS           200
GET_ANTIBIOTIC_INFO                200
GET_USES_INDICATIONS               200
GET_SUBSTANCE_INTERACTION          200
REDIRECT_MEDICINE_QUERY            200
GET_FOOD_AND_TIMING                199
Name: count, dtype: int64

Total samples : 1801
Unique intents: 9


In [5]:
# encode labels and split data for training and testing
le = LabelEncoder()
y = le.fit_transform(df["label"])
X = df["text"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"\nTrain size: {len(X_train)} | Test size: {len(X_test)}")


Train size: 1440 | Test size: 361


In [ ]:
#TF-IDF + SVM pipeline
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),      # unigrams + bigrams
        sublinear_tf=True,       # apply log normalization
        min_df=1,
        max_df=0.95,
    )),
    ("svm", SVC(
        kernel="linear",
        probability=True,
        random_state=42,
    )),
])


# hyperparameter tuning
param_grid = {
    "tfidf__max_features": [3000, 5000, None],
    "svm__C":              [0.1, 1, 10],
    "svm__kernel":         ["linear", "rbf"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=cv,
    scoring="f1_weighted",
    n_jobs=-1,
    verbose=2,
)

print("\nRunning Grid Search (this may take a few minutes)…")
grid_search.fit(X_train, y_train)

print("\nBest parameters :", grid_search.best_params_)
print("Best CV F1 score:", round(grid_search.best_score_, 4))

best_model = grid_search.best_estimator_


Running Grid Search (this may take a few minutes)…
Fitting 5 folds for each of 18 candidates, totalling 90 fits


In [ ]:
y_pred = best_model.predict(X_test)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_,
    zero_division=0,
))

In [ ]:
# confusion matrix
TOP_N = 15

# Get indices of top-N classes by frequency in test set
top_classes_idx = np.argsort(np.bincount(y_test))[-TOP_N:]
top_class_names  = le.classes_[top_classes_idx]

mask = np.isin(y_test, top_classes_idx)
cm   = confusion_matrix(y_test[mask], y_pred[mask], labels=top_classes_idx)

fig, ax = plt.subplots(figsize=(14, 11))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top_class_names)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap="Blues")
ax.set_title(f"Confusion Matrix per Class", fontsize=14, pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# f1 bar chart
from sklearn.metrics import f1_score

f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0, labels=top_classes_idx)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_class_names, f1_per_class, color="steelblue")
ax.set_xlabel("F1 Score")
ax.set_title("Per Class F1 Score")
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, f1_per_class):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("f1_per_class.png", dpi=150)
plt.show()

In [ ]:
sample_queries = [
    "What are the side effects of amoxicillin?",
    "How should I store ciprofloxacin?",
    "What is azithromycin used for?",
    "Can I take doxycycline with milk?",
    "What is the dosage of levofloxacin?",
]

print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)
preds = best_model.predict(sample_queries)
probs = best_model.predict_proba(sample_queries)

for q, pred, prob in zip(sample_queries, preds, probs):
    intent     = le.inverse_transform([pred])[0]
    confidence = prob.max()
    print(f"Query      : {q}")
    print(f"Predicted  : {intent}  (confidence: {confidence:.2%})")
    print("-" * 50)

# **DistilBERT**

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
)

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# import data
df = pd.read_csv('patient queries.csv')

In [ ]:
# preprocessing
QUESTION_COL = "Question"
INTENT_COL   = "Manual Validation"

df = df[[QUESTION_COL, INTENT_COL]].dropna()
df.columns = ["text", "label"]

df["text"] = (
    df["text"]
    .str.lower()
    .str.strip()
    .str.replace(r'["\']', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

print("\nClass distribution:")
print(df["label"].value_counts())
print(f"\nTotal samples : {len(df)}")
print(f"Unique intents: {df['label'].nunique()}")

# Encode labels
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])
NUM_LABELS = len(le.classes_)
print(f"\nNumber of intent classes: {NUM_LABELS}")

In [ ]:
# pytorch dataset class

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(list(labels), dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

In [ ]:
# training and evaluation helpers
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds        = logits.argmax(dim=-1)
        correct     += (preds == labels).sum().item()
        total       += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels         = batch["labels"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc      = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)

In [ ]:
# train final model  (train/val/test split)
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 128
BATCH_SIZE   = 16
EPOCHS       = 10
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
PATIENCE     = 3        # early stopping patience

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

X = df["text"].values
y = df["label_id"].values

# 80 / 10 / 10 split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

train_dataset = IntentDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = IntentDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = IntentDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

# Model, optimizer, scheduler
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

# Training loop with early stopping
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}
best_val_f1   = 0.0
patience_ctr  = 0
best_model_path = "best_distilbert_model.pt"

print("\nStarting training…")
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)
    val_loss, val_acc, val_f1, _, _ = eval_epoch(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}"
    )

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_ctr = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  Best model saved (val F1: {best_val_f1:.4f})")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stopping triggered after {epoch} epochs.")
            break

# ── Reload best checkpoint for evaluation ──
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
print(f"\nLoaded best model (val F1: {best_val_f1:.4f})")

In [ ]:
# test set evaluation
_, test_acc, test_f1, y_pred, y_true = eval_epoch(model, test_loader)

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
print(f"Accuracy : {test_acc:.4f}")
print(f"Macro F1 : {test_f1:.4f}\n")
print(classification_report(
    y_true, y_pred,
    target_names=le.classes_,
    zero_division=0,
))

In [ ]:
# k-fold cross validation
N_SPLITS    = 5
CV_EPOCHS   = 5     # fewer epochs per fold to keep runtime reasonable
fold_results = {"acc": [], "f1": []}

print(f"\nRunning {N_SPLITS}-Fold Stratified Cross-Validation…")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_f_train, X_f_val = X[train_idx], X[val_idx]
    y_f_train, y_f_val = y[train_idx], y[val_idx]

    f_train_ds = IntentDataset(X_f_train, y_f_train, tokenizer, MAX_LEN)
    f_val_ds   = IntentDataset(X_f_val,   y_f_val,   tokenizer, MAX_LEN)
    f_train_dl = DataLoader(f_train_ds, batch_size=BATCH_SIZE, shuffle=True)
    f_val_dl   = DataLoader(f_val_ds,   batch_size=BATCH_SIZE)

    f_model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    ).to(DEVICE)

    f_optimizer  = AdamW(f_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    f_total      = len(f_train_dl) * CV_EPOCHS
    f_warmup     = int(f_total * WARMUP_RATIO)
    f_scheduler  = get_linear_schedule_with_warmup(f_optimizer, f_warmup, f_total)

    for ep in range(CV_EPOCHS):
        train_epoch(f_model, f_train_dl, f_optimizer, f_scheduler)

    _, f_acc, f_f1, _, _ = eval_epoch(f_model, f_val_dl)
    fold_results["acc"].append(f_acc)
    fold_results["f1"].append(f_f1)
    print(f"  Fold {fold}: Acc={f_acc:.4f}  Macro-F1={f_f1:.4f}")

    del f_model
    torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

print(f"\nCross-Validation Summary ({N_SPLITS} folds):")
print(f"  Accuracy : {np.mean(fold_results['acc']):.4f} ± {np.std(fold_results['acc']):.4f}")
print(f"  Macro F1 : {np.mean(fold_results['f1']):.4f} ± {np.std(fold_results['f1']):.4f}")

In [ ]:
# plots
# Training curves
epochs_ran = len(history["train_loss"])
ep_range   = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(ep_range, history["train_loss"], label="Train", marker="o")
axes[0].plot(ep_range, history["val_loss"],   label="Val",   marker="o")
axes[0].set_title("Loss per Epoch"); axes[0].set_xlabel("Epoch"); axes[0].legend()

axes[1].plot(ep_range, history["train_acc"], label="Train", marker="o")
axes[1].plot(ep_range, history["val_acc"],   label="Val",   marker="o")
axes[1].set_title("Accuracy per Epoch"); axes[1].set_xlabel("Epoch"); axes[1].legend()

axes[2].plot(ep_range, history["val_f1"], color="green", marker="o", label="Val Macro-F1")
axes[2].set_title("Validation Macro-F1"); axes[2].set_xlabel("Epoch"); axes[2].legend()

plt.suptitle("DistilBERT Training History", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# confusion matrix
TOP_N = 15
top_classes_idx  = np.argsort(np.bincount(y_true))[-TOP_N:]
top_class_names  = le.classes_[top_classes_idx]

mask = np.isin(y_true, top_classes_idx)
cm   = confusion_matrix(y_true[mask], y_pred[mask], labels=top_classes_idx)

fig, ax = plt.subplots(figsize=(14, 11))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top_class_names)
disp.plot(ax=ax, xticks_rotation=45, colorbar=True, cmap="Blues")
ax.set_title(f"Confusion Matrix – Top {TOP_N} Intent Classes (DistilBERT)", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig("confusion_matrix_distilbert.png", dpi=150)
plt.show()

In [ ]:
# f1 bar chart
f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0, labels=top_classes_idx)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_class_names, f1_per_class, color="steelblue")
ax.set_xlabel("F1 Score"); ax.set_title("Per-Class F1 Score – Top 15 Intents (DistilBERT)")
ax.set_xlim(0, 1.05)
for bar, val in zip(bars, f1_per_class):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("f1_per_class_distilbert.png", dpi=150)
plt.show()
print("F1 chart saved → f1_per_class_distilbert.png")

# K-Fold results bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x     = np.arange(N_SPLITS)
width = 0.35
ax.bar(x - width/2, fold_results["acc"], width, label="Accuracy", color="steelblue")
ax.bar(x + width/2, fold_results["f1"],  width, label="Macro-F1", color="coral")
ax.set_xticks(x); ax.set_xticklabels([f"Fold {i+1}" for i in range(N_SPLITS)])
ax.set_ylim(0, 1.1); ax.set_title("K-Fold Cross-Validation Results (DistilBERT)")
ax.legend(); ax.set_ylabel("Score")
plt.tight_layout()
plt.savefig("kfold_results_distilbert.png", dpi=150)
plt.show()